### Test ydadta profiling (P94)

In [0]:
%pip install --upgrade ydata_profiling typing_extensions databricks-automl-runtime
dbutils.library.restartPython()

In [0]:
import pandas as pd
from ydata_profiling import ProfileReport

df = pd.read_csv("/Volumes/porya_catalog/default/favorita_forecasting/raw_data/store-sales-time-series-forecasting/test.csv")
profile = ProfileReport(df, minimal=True, title="My Report")
# profile.to_file("report.html")  # save HTML
# or, in a notebook:
profile.to_notebook_iframe()

## Chapter 4-1: Getting to Know Your Data/Project:
* Favorita Store Sales - Time Series Forecasting/CH4-01-Exploring Favorita Sales Data.sql

NOTE: as we use serverless compute, it is not working. instead we use ydata profiling library

In [0]:
# Create a Unity Catalog Volume
spark.sql("CREATE VOLUME IF NOT EXISTS porya_catalog.default.favorita_forecasting")
# Sets the file path variable 
volume_file_path = "/Volumes/porya_catalog/default/favorita_forecasting/"

In [0]:
%sql
SELECT * FROM porya_catalog.default.train_set

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT ts.* FROM porya_catalog.default.train_set ts 
INNER JOIN (SELECT family, sum(sales) as total_sales FROM porya_catalog.default.train_set GROUP BY family ORDER BY total_sales DESC LIMIT 10) top_10 ON ts.family=top_10.family

## Chapter 4-2:
* Getting to Know Your Data/Project: Favorita Store Sales - Time Series Forecasting/CH4-02-Exploring Autogenerated Notebook.py

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Data Exploration
# MAGIC - This notebook performs exploratory data analysis on the dataset.
# MAGIC - To expand on the analysis, attach this notebook to a cluster with runtime version **14.0.x-snapshot-cpu-ml-scala2.12**,
# MAGIC edit [the options of pandas-profiling](https://pandas-profiling.ydata.ai/docs/master/rtd/pages/advanced_usage.html), and rerun it.
# MAGIC - Explore completed trials in the [MLflow experiment](#mlflow/experiments/1430361843542082).

# COMMAND ----------

import mlflow
import os
import uuid
import shutil
import pandas as pd
import databricks.automl_runtime

# Load training data directly from Unity Catalog table
df = spark.table("porya_catalog.default.train_set").drop("id").toPandas()

target_col = "sales"

# Convert Spark date columns to Pandas datetime
date_columns = ['date']
df[date_columns] = df[date_columns].apply(pd.to_datetime, errors="coerce")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Truncate rows
# MAGIC Only the first 10000 rows will be considered for pandas-profiling to avoid out-of-memory issues.
# MAGIC Comment out next cell and rerun the notebook to profile the full dataset.

# COMMAND ----------

df = df.iloc[:10000, :]

# COMMAND ----------

# MAGIC %md
# MAGIC ## Profiling Results

# COMMAND ----------

from ydata_profiling import ProfileReport
df_profile = ProfileReport(df,
                           correlations={
                               "auto": {"calculate": True},
                               "pearson": {"calculate": True},
                               "spearman": {"calculate": True},
                               "kendall": {"calculate": True},
                               "phi_k": {"calculate": True},
                               "cramers": {"calculate": True},
                           }, title="Profiling Report", progress_bar=False, infer_dtypes=False)
profile_html = df_profile.to_html()

displayHTML(profile_html)

> ## Chapter 4-3: third file of the book
*  Getting to Know Your Data/Project: Favorita Store Sales - Time Series Forecasting/CH4-03-Imputing Oil Data.py

In [0]:
import pyspark.pandas as ps

df = ps.read_table("porya_catalog.default.oil_prices", index_col="date")
df.head(10)

# COMMAND ----------

# DBTITLE 1,Write dataframe with imputed data to silver table
df = (
    df.reindex(ps.date_range(df.index.min(), df.index.max()))
    .reset_index()
    .rename(columns={"index": "date"})
    .ffill()
)
df.to_table("porya_catalog.default.oil_prices_silver")
df.head(10)